In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None) # Show all columns
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup Complete.")

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Setup Complete.


In [2]:
# Find file paths
train_path, test_path, sub_path = '', '', ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full = os.path.join(dirname, filename)
        if 'train.csv' in filename: train_path = full
        elif 'test.csv' in filename: test_path = full
        elif 'sample_submission.csv' in filename: sub_path = full

# Load Data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(sub_path)

print(f"Data Loaded.")
print(f"Train: {train.shape}, Test: {test.shape}")

Data Loaded.
Train: (700000, 26), Test: (300000, 25)


In [3]:
# v2 : Upgrade Feature
def engineer_features(df):
    # 1. Pulse Pressure (Systolic - Diastolic)
    # High pulse pressure is a risk factor for diabetes complications
    df['Pulse_Pressure'] = df['systolic_bp'] - df['diastolic_bp']
    
    # 2. Mean Arterial Pressure (MAP)
    df['MAP'] = df['diastolic_bp'] + (df['Pulse_Pressure'] / 3)
    
    # 3. Cholesterol Ratio (Total / HDL) - Atherogenic Index
    # Very important cardiovascular risk marker
    # Add small epsilon (1e-6) to avoid division by zero
    df['Cholesterol_Ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1e-6)
    
    # 4. BMI Categories (0: Underweight, 1: Normal, 2: Overweight, 3: Obese)
    df['BMI_Cat'] = pd.cut(df['bmi'], 
                           bins=[0, 18.5, 24.9, 29.9, 100], 
                           labels=[0, 1, 2, 3]).astype(int)
    
    # 5. Risk Interaction (Age * BMI)
    # Older and higher BMI = Higher Risk
    df['Age_BMI_Interaction'] = df['age'] * df['bmi']
    
    # 6. Triglyceride to HDL Ratio (Indicator of Insulin Resistance)
    df['TG_HDL_Ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-6)
    
    return df

print("Creating new features...")

# Combine Train/Test for consistent processing
train_len = len(train)
target = train['diagnosed_diabetes']
train = train.drop('diagnosed_diabetes', axis=1)

# Drop ID
test_ids = test['id']
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

# Apply Feature Engineering
combined = pd.concat([train, test], axis=0)
combined = engineer_features(combined)

print("Feature Engineering Complete.")
display(combined.head(3))

Creating new features...
Feature Engineering Complete.


,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,Pulse_Pressure,MAP,Cholesterol_Ratio,BMI_Cat,Age_BMI_Interaction,TG_HDL_Ratio
0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,70,62,199,58,114,102,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,42,84.000000,3.431034,3,1035.4,1.758621
1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,77,71,199,50,121,124,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,43,91.333333,3.980000,1,1190.0,2.480000
2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,89,73,188,59,114,108,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,6,91.000000,3.186441,1,771.2,1.830508


In [4]:
# 1. One-Hot Encoding
combined_encoded = pd.get_dummies(combined, drop_first=True)

# 2. Split back to Train/Test
X = combined_encoded.iloc[:train_len]
X_test = combined_encoded.iloc[train_len:]

# 3. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

print(f"Data Ready for Modeling. Features: {X.shape[1]}")

Data Ready for Modeling. Features: 42


In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, target, test_size=0.2, random_state=42, stratify=target
)

print(f"Train Shape: {X_train.shape}")
print(f"Val Shape: {X_val.shape}")

Train Shape: (560000, 42)
Val Shape: (140000, 42)


In [6]:
# Initialize XGBoost (Added early_stopping_rounds in constructor)
xgb_model = XGBClassifier(
    n_estimators=2500,       # Increased trees
    learning_rate=0.015,     # Slower learning rate for precision
    max_depth=7,             # Slightly deeper trees
    subsample=0.7,
    colsample_bytree=0.7,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    device='cpu',            # Change to 'cuda' for GPU
    early_stopping_rounds=150
)

# Train
print("Training XGBoost...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=500
)

# Score
xgb_score = roc_auc_score(y_val, xgb_model.predict_proba(X_val)[:, 1])
print(f"\nXGBoost Validation AUC: {xgb_score:.5f}")

Training XGBoost...
[0]	validation_0-auc:0.69445	validation_1-auc:0.69260
[500]	validation_0-auc:0.73246	validation_1-auc:0.72032
[1000]	validation_0-auc:0.74643	validation_1-auc:0.72365
[1500]	validation_0-auc:0.75722	validation_1-auc:0.72496
[2000]	validation_0-auc:0.76700	validation_1-auc:0.72560
[2499]	validation_0-auc:0.77570	validation_1-auc:0.72579

XGBoost Validation AUC: 0.72581


In [7]:
from lightgbm import early_stopping, log_evaluation

# Initialize LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=2500,
    learning_rate=0.015,
    num_leaves=40,           # Increased complexity
    subsample=0.7,
    colsample_bytree=0.7,
    metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Train
print("Training LightGBM...")
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    callbacks=[early_stopping(150), log_evaluation(500)]
)

# Score
lgbm_score = roc_auc_score(y_val, lgbm_model.predict_proba(X_val)[:, 1])
print(f"\nLightGBM Validation AUC: {lgbm_score:.5f}")

Training LightGBM...
Training until validation scores don't improve for 150 rounds
[500]	training's auc: 0.727105	valid_1's auc: 0.722511
[1000]	training's auc: 0.734438	valid_1's auc: 0.724573
[1500]	training's auc: 0.740288	valid_1's auc: 0.725367
[2000]	training's auc: 0.745691	valid_1's auc: 0.725928
[2500]	training's auc: 0.750782	valid_1's auc: 0.726219
Did not meet early stopping. Best iteration is:
[2496]	training's auc: 0.750742	valid_1's auc: 0.726221

LightGBM Validation AUC: 0.72622


In [8]:
# 1. Predictions on Test Set
# Use best iteration for XGBoost
xgb_pred = xgb_model.predict_proba(X_test_scaled, iteration_range=(0, xgb_model.best_iteration + 1))[:, 1]
lgbm_pred = lgbm_model.predict_proba(X_test_scaled)[:, 1]

# 2. Weighted Ensemble
# If one model is significantly better, increase its weight.
# Default: 0.5 vs 0.5
w_xgb = 0.5
w_lgbm = 0.5

final_pred = (xgb_pred * w_xgb) + (lgbm_pred * w_lgbm)

# 3. Create Submission File
submission_df = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': final_pred
})

submission_df.to_csv('submission.csv', index=False)

print("'submission.csv(v2)' Created!")
print(f"Weights used -> XGB: {w_xgb}, LGBM: {w_lgbm}")
display(submission_df.head())

'submission.csv(v2)' Created!
Weights used -> XGB: 0.5, LGBM: 0.5


,id,diagnosed_diabetes
0,700000,0.490603
1,700001,0.673961
2,700002,0.781673
3,700003,0.405449
4,700004,0.922402
